# CodeGauge: Programming Problem Difficulty Classification
**End-to-End Machine Learning Workflow in Snowflake**

This notebook classifies Codeforces programming problems as **Easy**, **Medium**, or **Hard** using aggregate submission behavior — without using the official problem rating as a model input.

### Workflow
1. **Data Exploration** — Inspect raw submission data (14.9M rows)
2. **Feature Engineering** — Aggregate per-problem submission metrics
3. **Target Creation** — Derive difficulty labels from problem ratings
4. **Train/Test Split** — Stratified 50/50 split
5. **Model Training** — Snowflake-native Classification
6. **Prediction & Evaluation** — Accuracy, confusion matrix, feature importance

## 1. Data Exploration
Raw data: **14.9M Codeforces submissions** with user handle, rating, problem info, and verdict.

In [ ]:
%%sql -r raw_data
-- Raw submission data: 14.9M rows from Codeforces
SELECT * FROM CODEGAUGE_DB.ML.CODEGAUGE_DATA LIMIT 10;

In [ ]:
%%sql -r data_summary
-- How many total submissions and unique problems?
SELECT 
    COUNT(*) AS TOTAL_SUBMISSIONS,
    COUNT(DISTINCT CONTESTID || '-' || INDEX) AS UNIQUE_PROBLEMS,
    COUNT(DISTINCT HANDLE) AS UNIQUE_USERS
FROM CODEGAUGE_DB.ML.CODEGAUGE_DATA;

In [ ]:
%%sql -r verdict_dist
-- Verdict distribution across all submissions
SELECT 
    VERDICT, 
    COUNT(*) AS CNT,
    ROUND(CNT / SUM(CNT) OVER () * 100, 2) AS PCT
FROM CODEGAUGE_DB.ML.CODEGAUGE_DATA
GROUP BY VERDICT
ORDER BY CNT DESC;

## 2. Feature Engineering
Aggregate 14.9M submissions into **per-problem behavioral features**:
- `TOTAL_SUBMISSIONS` — total attempts on the problem
- `ACCEPTED_SUBMISSIONS` — how many got accepted (OK)
- `WRONG_ANSWERS` — how many got wrong answer
- `UNIQUE_USERS` — distinct users who attempted
- `ACCEPTANCE_RATE` — accepted / total

Plus contextual features: `CONTESTID`, `INDEX` (problem position A-F), `TAGS` (topic tags).

In [ ]:
%%sql -r features_created
-- Aggregate raw submissions into per-problem features
CREATE OR REPLACE TABLE CODEGAUGE_DB.ML.PROBLEM_FEATURES AS
SELECT 
    CONTESTID,
    INDEX,
    NAME,
    TAGS,
    MAX(PROBLEM_RATING) AS PROBLEM_RATING,
    COUNT(*) AS TOTAL_SUBMISSIONS,
    COUNT_IF(VERDICT = 'OK') AS ACCEPTED_SUBMISSIONS,
    COUNT_IF(VERDICT = 'WRONG_ANSWER') AS WRONG_ANSWERS,
    COUNT(DISTINCT HANDLE) AS UNIQUE_USERS,
    ROUND(ACCEPTED_SUBMISSIONS / NULLIF(TOTAL_SUBMISSIONS, 0), 6) AS ACCEPTANCE_RATE
FROM CODEGAUGE_DB.ML.CODEGAUGE_DATA
GROUP BY CONTESTID, INDEX, NAME, TAGS;

In [ ]:
%%sql -r features_preview
-- Preview engineered features
SELECT * FROM CODEGAUGE_DB.ML.PROBLEM_FEATURES LIMIT 10;

## 3. Target Creation
Derive difficulty labels from the official `PROBLEM_RATING`:
- **Easy**: rating 800–1200
- **Medium**: rating 1300–2000  
- **Hard**: rating 2100–3500

Note: `PROBLEM_RATING` is used ONLY to create the label. It is **excluded** from model features to avoid data leakage.

In [ ]:
%%sql -r split_created
-- Add difficulty label and perform stratified 50/50 train/test split
CREATE OR REPLACE TABLE CODEGAUGE_DB.ML.ML_SPLIT AS
SELECT *,
    CASE 
        WHEN PROBLEM_RATING BETWEEN 800 AND 1200 THEN 'Easy'
        WHEN PROBLEM_RATING BETWEEN 1300 AND 2000 THEN 'Medium'
        WHEN PROBLEM_RATING >= 2100 THEN 'Hard'
    END AS DIFFICULTY,
    CASE 
        WHEN ROW_NUMBER() OVER (PARTITION BY DIFFICULTY ORDER BY RANDOM()) 
             <= COUNT(*) OVER (PARTITION BY DIFFICULTY) * 0.5 
        THEN 'TRAIN' ELSE 'TEST' 
    END AS DATASET
FROM CODEGAUGE_DB.ML.PROBLEM_FEATURES
WHERE PROBLEM_RATING IS NOT NULL;

In [ ]:
%%sql -r split_balance
-- Verify split balance
SELECT DATASET, DIFFICULTY, COUNT(*) AS CNT
FROM CODEGAUGE_DB.ML.ML_SPLIT
GROUP BY DATASET, DIFFICULTY
ORDER BY DATASET, DIFFICULTY;

## 4. Prepare Clean Training Data
Drop `PROBLEM_RATING` (data leakage) but keep legitimate features:
- **Behavioral**: submission counts, acceptance rate, wrong rate, user ratios
- **Contextual**: CONTESTID (contest era), INDEX (problem position A-F), TAGS (topic complexity)
- **Derived**: WRONG_RATE, USER_SUBMISSION_RATIO, OTHER_VERDICTS, TAG_COUNT, WRONG_PER_USER, SUBMISSIONS_PER_USER

In [ ]:
-- Create clean training set: behavioral + contextual + derived features
CREATE OR REPLACE TABLE CODEGAUGE_DB.ML.ML_TRAIN_CLEAN AS
SELECT 
    CONTESTID,
    INDEX,
    TAGS,
    TOTAL_SUBMISSIONS,
    ACCEPTED_SUBMISSIONS,
    WRONG_ANSWERS,
    UNIQUE_USERS,
    ACCEPTANCE_RATE,
    ROUND(WRONG_ANSWERS / NULLIF(TOTAL_SUBMISSIONS, 0), 6) AS WRONG_RATE,
    ROUND(UNIQUE_USERS / NULLIF(TOTAL_SUBMISSIONS, 0), 6) AS USER_SUBMISSION_RATIO,
    TOTAL_SUBMISSIONS - ACCEPTED_SUBMISSIONS - WRONG_ANSWERS AS OTHER_VERDICTS,
    ARRAY_SIZE(SPLIT(TAGS, ',')) AS TAG_COUNT,
    ROUND(WRONG_ANSWERS / NULLIF(UNIQUE_USERS, 0), 6) AS WRONG_PER_USER,
    ROUND(TOTAL_SUBMISSIONS / NULLIF(UNIQUE_USERS, 0), 6) AS SUBMISSIONS_PER_USER,
    DIFFICULTY
FROM CODEGAUGE_DB.ML.ML_SPLIT
WHERE DATASET = 'TRAIN';

In [ ]:
%%sql -r test_created
-- Create test set with problem identifiers for result inspection
CREATE OR REPLACE TABLE CODEGAUGE_DB.ML.ML_TEST AS
SELECT 
    CONTESTID, INDEX, NAME, TAGS, PROBLEM_RATING,
    TOTAL_SUBMISSIONS, ACCEPTED_SUBMISSIONS, WRONG_ANSWERS,
    UNIQUE_USERS, ACCEPTANCE_RATE, DIFFICULTY
FROM CODEGAUGE_DB.ML.ML_SPLIT
WHERE DATASET = 'TEST';

## 5. Model Training
Train a **Snowflake-native Classification model** using `SNOWFLAKE.ML.CLASSIFICATION`.

This uses gradient-boosted trees under the hood — no external libraries needed.

In [ ]:
%%sql -r model_trained
-- Train the classification model
CREATE OR REPLACE SNOWFLAKE.ML.CLASSIFICATION CODEGAUGE_DB.ML.CODEGAUGE_CLASSIFIER(
    INPUT_DATA => SYSTEM$REFERENCE('TABLE', 'CODEGAUGE_DB.ML.ML_TRAIN_CLEAN'),
    TARGET_COLNAME => 'DIFFICULTY'
);

## 6. Prediction
Run the trained model against the held-out test set.

In [ ]:
-- Generate predictions on the test set
CREATE OR REPLACE TABLE CODEGAUGE_DB.ML.ML_PREDICTIONS AS
SELECT 
    t.CONTESTID,
    t.INDEX,
    t.NAME,
    t.DIFFICULTY AS ACTUAL_DIFFICULTY,
    CODEGAUGE_DB.ML.CODEGAUGE_CLASSIFIER!PREDICT(
        INPUT_DATA => OBJECT_CONSTRUCT(
            'CONTESTID', t.CONTESTID,
            'INDEX', t.INDEX,
            'TAGS', t.TAGS,
            'TOTAL_SUBMISSIONS', t.TOTAL_SUBMISSIONS,
            'ACCEPTED_SUBMISSIONS', t.ACCEPTED_SUBMISSIONS,
            'WRONG_ANSWERS', t.WRONG_ANSWERS,
            'UNIQUE_USERS', t.UNIQUE_USERS,
            'ACCEPTANCE_RATE', t.ACCEPTANCE_RATE,
            'WRONG_RATE', ROUND(t.WRONG_ANSWERS / NULLIF(t.TOTAL_SUBMISSIONS, 0), 6),
            'USER_SUBMISSION_RATIO', ROUND(t.UNIQUE_USERS / NULLIF(t.TOTAL_SUBMISSIONS, 0), 6),
            'OTHER_VERDICTS', t.TOTAL_SUBMISSIONS - t.ACCEPTED_SUBMISSIONS - t.WRONG_ANSWERS,
            'TAG_COUNT', ARRAY_SIZE(SPLIT(t.TAGS, ',')),
            'WRONG_PER_USER', ROUND(t.WRONG_ANSWERS / NULLIF(t.UNIQUE_USERS, 0), 6),
            'SUBMISSIONS_PER_USER', ROUND(t.TOTAL_SUBMISSIONS / NULLIF(t.UNIQUE_USERS, 0), 6)
        )
    ):class::VARCHAR AS PREDICTED_DIFFICULTY
FROM CODEGAUGE_DB.ML.ML_TEST t;

In [ ]:
%%sql -r pred_preview
-- Preview: Actual vs Predicted
SELECT * FROM CODEGAUGE_DB.ML.ML_PREDICTIONS LIMIT 15;

## 7. Model Evaluation
Compute accuracy, per-class precision/recall/F1, and visualize with a confusion matrix.

In [ ]:
%%sql -r accuracy
-- Overall Accuracy
SELECT 
    COUNT(*) AS TOTAL_PREDICTIONS,
    SUM(CASE WHEN ACTUAL_DIFFICULTY = PREDICTED_DIFFICULTY THEN 1 ELSE 0 END) AS CORRECT,
    ROUND(CORRECT / TOTAL_PREDICTIONS * 100, 2) AS ACCURACY_PCT
FROM CODEGAUGE_DB.ML.ML_PREDICTIONS;

In [ ]:
%%sql -r eval_metrics
-- Per-class evaluation metrics from the model
CALL CODEGAUGE_DB.ML.CODEGAUGE_CLASSIFIER!SHOW_EVALUATION_METRICS();

In [ ]:
%%sql -r confusion_data
-- Confusion Matrix: Actual vs Predicted counts
SELECT 
    ACTUAL_DIFFICULTY,
    PREDICTED_DIFFICULTY,
    COUNT(*) AS CNT
FROM CODEGAUGE_DB.ML.ML_PREDICTIONS
GROUP BY ACTUAL_DIFFICULTY, PREDICTED_DIFFICULTY
ORDER BY ACTUAL_DIFFICULTY, PREDICTED_DIFFICULTY;

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Get confusion matrix data
cm_df = confusion_data.to_pandas() if not isinstance(confusion_data, pd.DataFrame) else confusion_data.copy()

# Pivot into matrix form
labels = ['Easy', 'Medium', 'Hard']
cm = cm_df.pivot(index='ACTUAL_DIFFICULTY', columns='PREDICTED_DIFFICULTY', values='CNT').reindex(index=labels, columns=labels).fillna(0).astype(int)

# Plot confusion matrix
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(cm.values, interpolation='nearest', cmap='Blues')
ax.set_title('Confusion Matrix — CodeGauge Classifier', fontsize=14, fontweight='bold')
plt.colorbar(im, ax=ax)

tick_marks = np.arange(len(labels))
ax.set_xticks(tick_marks)
ax.set_xticklabels(labels, fontsize=12)
ax.set_yticks(tick_marks)
ax.set_yticklabels(labels, fontsize=12)
ax.set_xlabel('Predicted Difficulty', fontsize=12)
ax.set_ylabel('Actual Difficulty', fontsize=12)

thresh = cm.values.max() / 2
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, format(cm.values[i, j], 'd'),
                ha='center', va='center', fontsize=14,
                color='white' if cm.values[i, j] > thresh else 'black')

plt.tight_layout()
plt.show()

In [ ]:
# Per-class F1 scores bar chart
metrics_df = eval_metrics.to_pandas() if not isinstance(eval_metrics, pd.DataFrame) else eval_metrics.copy()
f1_df = metrics_df[metrics_df['ERROR_METRIC'] == 'f1'][['CLASS', 'METRIC_VALUE']].copy()
f1_df['METRIC_VALUE'] = f1_df['METRIC_VALUE'].astype(float)

colors = {'Easy': '#2ecc71', 'Medium': '#f39c12', 'Hard': '#e74c3c'}
bar_colors = [colors.get(c, '#3498db') for c in f1_df['CLASS']]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(f1_df['CLASS'], f1_df['METRIC_VALUE'], color=bar_colors, edgecolor='black', linewidth=0.8)
for bar, val in zip(bars, f1_df['METRIC_VALUE']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{val:.3f}',
            ha='center', va='bottom', fontsize=13, fontweight='bold')

ax.set_ylim(0, 1.0)
ax.set_ylabel('F1 Score', fontsize=12)
ax.set_title('F1 Score by Difficulty Class', fontsize=14, fontweight='bold')
ax.axhline(y=0.7, color='gray', linestyle='--', alpha=0.5, label='0.70 baseline')
ax.legend()
plt.tight_layout()
plt.show()

## 8. Feature Importance
Which submission metrics matter most for predicting difficulty?

In [ ]:
%%sql -r feature_importance
-- Feature importance from the trained model
CALL CODEGAUGE_DB.ML.CODEGAUGE_CLASSIFIER!SHOW_FEATURE_IMPORTANCE();

In [ ]:
# Feature importance visualization
fi_df = feature_importance.to_pandas() if not isinstance(feature_importance, pd.DataFrame) else feature_importance.copy()
fi_df['SCORE'] = fi_df['SCORE'].astype(float)
fi_df = fi_df.sort_values('SCORE', ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
colors_fi = plt.cm.viridis(np.linspace(0.3, 0.9, len(fi_df)))
bars = ax.barh(fi_df['FEATURE'], fi_df['SCORE'], color=colors_fi, edgecolor='black', linewidth=0.5)

for bar, val in zip(bars, fi_df['SCORE']):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.1%}', ha='left', va='center', fontsize=11)

ax.set_xlabel('Importance Score', fontsize=12)
ax.set_title('Feature Importance — CodeGauge Classifier', fontsize=14, fontweight='bold')
ax.set_xlim(0, max(fi_df['SCORE']) * 1.2)
plt.tight_layout()
plt.show()

## 9. Key Findings

| Insight | Detail |
|---|---|
| **Input** | 14 features: behavioral (submissions, acceptance rate) + contextual (contest ID, index, tags) + derived ratios |
| **Output** | Difficulty class: Easy, Medium, or Hard |
| **Accuracy** | ~82% on held-out test set (up from 69% with behavioral-only) |
| **Best classified** | Hard (F1~0.88) — fewer users attempt, more wrong answers, later index position |
| **Hardest to classify** | Medium (F1~0.77) — boundary class between Easy and Hard |
| **Top features** | TAGS (15%), CONTESTID (13%), UNIQUE_USERS (10%), INDEX (8%) |
| **No leakage** | PROBLEM_RATING excluded from features; model learns from behavior + context |

In [ ]:
-- Quick demo: Predict difficulty for a hypothetical problem
-- Low acceptance rate (15%) + many wrong answers + late index (E) = likely Hard
SELECT 
    'Demo Problem' AS PROBLEM,
    CODEGAUGE_DB.ML.CODEGAUGE_CLASSIFIER!PREDICT(
        INPUT_DATA => OBJECT_CONSTRUCT(
            'CONTESTID', 1900,
            'INDEX', 'E',
            'TAGS', '[''dp'', ''math'', ''number theory'']',
            'TOTAL_SUBMISSIONS', 8000,
            'ACCEPTED_SUBMISSIONS', 1200,
            'WRONG_ANSWERS', 5500,
            'UNIQUE_USERS', 3000,
            'ACCEPTANCE_RATE', 0.15,
            'WRONG_RATE', 0.6875,
            'USER_SUBMISSION_RATIO', 0.375,
            'OTHER_VERDICTS', 1300,
            'TAG_COUNT', 3,
            'WRONG_PER_USER', 1.833333,
            'SUBMISSIONS_PER_USER', 2.666667
        )
    ) AS PREDICTION;

## 10. Interactive Prediction Tool
Enter a **problem name** or **contest ID** below and click **Predict** to:
1. Look up the problem's submission features
2. Predict difficulty using the trained model
3. Cross-check against actual difficulty

In [ ]:
import ipywidgets as widgets
import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()

input_box = widgets.Text(
    value='',
    placeholder='e.g. "Binary Search" or "1945"',
    description='',
    layout=widgets.Layout(width='400px')
)

predict_btn = widgets.Button(
    description='Predict',
    button_style='info',
    layout=widgets.Layout(width='120px')
)

output = widgets.Output()

def on_predict(btn):
    output.clear_output()
    user_input = input_box.value.strip()
    if not user_input:
        with output:
            print("Please enter a problem name or contest ID.")
        return

    with output:
        print(f"Searching for '{user_input}'...\n")

    safe_input = user_input.replace("'", "''")

    if safe_input.isdigit():
        where_clause = f"s.CONTESTID = {safe_input}"
    else:
        where_clause = f"s.NAME ILIKE '%{safe_input}%'"

    query = f"""
    SELECT 
        s.NAME AS PROBLEM_NAME,
        s.CONTESTID,
        s.INDEX,
        s.TAGS,
        s.TOTAL_SUBMISSIONS,
        s.ACCEPTED_SUBMISSIONS,
        s.WRONG_ANSWERS,
        s.UNIQUE_USERS,
        s.ACCEPTANCE_RATE,
        s.DIFFICULTY AS ACTUAL_DIFFICULTY,
        CODEGAUGE_DB.ML.CODEGAUGE_CLASSIFIER!PREDICT(
            INPUT_DATA => OBJECT_CONSTRUCT(
                'CONTESTID', s.CONTESTID,
                'INDEX', s.INDEX,
                'TAGS', s.TAGS,
                'TOTAL_SUBMISSIONS', s.TOTAL_SUBMISSIONS,
                'ACCEPTED_SUBMISSIONS', s.ACCEPTED_SUBMISSIONS,
                'WRONG_ANSWERS', s.WRONG_ANSWERS,
                'UNIQUE_USERS', s.UNIQUE_USERS,
                'ACCEPTANCE_RATE', s.ACCEPTANCE_RATE,
                'WRONG_RATE', ROUND(s.WRONG_ANSWERS / NULLIF(s.TOTAL_SUBMISSIONS, 0), 6),
                'USER_SUBMISSION_RATIO', ROUND(s.UNIQUE_USERS / NULLIF(s.TOTAL_SUBMISSIONS, 0), 6),
                'OTHER_VERDICTS', s.TOTAL_SUBMISSIONS - s.ACCEPTED_SUBMISSIONS - s.WRONG_ANSWERS,
                'TAG_COUNT', ARRAY_SIZE(SPLIT(s.TAGS, ',')),
                'WRONG_PER_USER', ROUND(s.WRONG_ANSWERS / NULLIF(s.UNIQUE_USERS, 0), 6),
                'SUBMISSIONS_PER_USER', ROUND(s.TOTAL_SUBMISSIONS / NULLIF(s.UNIQUE_USERS, 0), 6)
            )
        ):class::VARCHAR AS PREDICTED_DIFFICULTY,
        CASE 
            WHEN s.DIFFICULTY = PREDICTED_DIFFICULTY THEN 'CORRECT'
            ELSE 'WRONG'
        END AS VERDICT
    FROM CODEGAUGE_DB.ML.ML_SPLIT s
    WHERE {where_clause}
    ORDER BY s.TOTAL_SUBMISSIONS DESC
    LIMIT 10
    """

    try:
        df = session.sql(query).to_pandas()
    except Exception as e:
        with output:
            output.clear_output()
            print(f"Error: {e}")
        return

    output.clear_output()
    with output:
        if df.empty:
            print(f"No problems found matching '{user_input}'.")
            print("Try a different name (e.g. 'Two Sum', 'Watermelon') or contest ID (e.g. '1945').")
            return

        correct = (df['VERDICT'] == 'CORRECT').sum()
        total = len(df)

        for _, row in df.iterrows():
            symbol = 'O' if row['VERDICT'] == 'CORRECT' else 'X'
            print(f"Problem   : {row['PROBLEM_NAME']} (Contest {row['CONTESTID']}{row['INDEX']})")
            print(f"Features  : {row['TOTAL_SUBMISSIONS']:,} submissions | {row['UNIQUE_USERS']:,} users | {row['ACCEPTANCE_RATE']:.1%} acceptance")
            print(f"Predicted : {row['PREDICTED_DIFFICULTY']}")
            print(f"Actual    : {row['ACTUAL_DIFFICULTY']}")
            print(f"Result    : [{symbol}] {row['VERDICT']}")
            print("-" * 60)

        print(f"\nModel got {correct}/{total} correct ({correct/total:.0%} accuracy on this search)")

predict_btn.on_click(on_predict)

prompt_label = widgets.HTML(value="<b>Enter a problem name or contest ID:</b>")
input_row = widgets.HBox([input_box, predict_btn])
display(widgets.VBox([prompt_label, input_row, output]))